# Week 11 - Interactive Visualization with Plotly

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 11 - Interactive Visualization**

---

## Session topics (from the course outline)

- Creating interactive charts
- Plotly Express versus Graph Objects
- Introduction to dashboards

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 3** | Apply Python libraries (NumPy, Pandas) to clean, manipulate and process datasets |
| **CLO 6** | Generate statistical summaries to describe data patterns |
| **CLO 8** | Evaluate the effectiveness of different visualization methods for communicating insights |

## Reading

| Topic | Source |
|---|---|
| Other Python visualization tools | McKinney §9.3 |
| Plotly Express documentation | `https://plotly.com/python/plotly-express/` |

## The problem interactivity solves

Week 10 ended on a difficulty. A year of daily data has 365 points; drawn at the width of a
page, they overlap into a band. Resampling to monthly makes it readable and destroys the
weekly cycle in the process.

Every static chart faces some version of that: the page is a fixed size, so detail and
overview compete for it. You choose one, and the reader gets what you chose.

Interactivity changes the arrangement. The reader zooms, hovers for exact values, and hides
series they do not care about — so the chart can hold both the overview and the detail, and
the reader decides which to look at.

That is genuinely useful and it is not free. This session covers what it costs, and when a
static chart remains the better answer.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Render Plotly figures inline, with the JavaScript embedded so they work
# without an internet connection. Alternatives:
#   "notebook_connected"  smaller output, loads plotly.js from the internet
#   "browser"             opens each figure in a new browser tab
#   "png"                 static images (needs the kaleido package)
pio.renderers.default = "notebook"

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 — Theory and concepts

## 1.1 What interactivity adds

| Interaction | What it lets the reader do |
|---|---|
| **Hover** | Read the exact value of a point without an axis lookup |
| **Zoom and pan** | Inspect a region without a second chart |
| **Legend click** | Hide and show series |
| **Selection** | Isolate a subset by dragging over it |

These address a specific limitation: a static chart has one level of detail, fixed by whoever
made it. Interactivity lets the chart hold several and lets the reader move between them.

## 1.2 What it costs

Being honest about the trade-off is most of the judgement in this session.

| Cost | Detail |
|---|---|
| **It needs a browser** | An interactive chart cannot go in a printed report, a PDF or a slide that will be projected from a file |
| **File size** | The data is embedded in the HTML. A million points is a very large file |
| **Nobody may interact** | Many readers look and move on. Anything that requires a hover to be understood has not been communicated |
| **It can hide the point** | A reader who must explore to find the finding may not find it |

**The rule that follows:** the chart must make sense **before** anyone touches it. Interactivity
is for the reader who wants to go further, not a substitute for a clear default view.

## 1.3 Plotly Express and Graph Objects

Plotly has two interfaces, and the relationship between them is the same as seaborn's to
matplotlib.

| | Plotly Express (`px`) | Graph Objects (`go`) |
|---|---|---|
| Input | A DataFrame plus column names | Explicit traces |
| Lines of code | One | Several |
| Control | What the function offers | Everything |
| Returns | A `Figure` | A `Figure` |

Both return the **same object**. So the normal approach is to build with `px` and then adjust
with the figure's own methods:

```python
fig = px.scatter(df, x="attendance", y="gpa")
fig.update_layout(title="...", height=400)
fig.update_traces(marker=dict(size=8))
```

Reach for `go` when you need something `px` does not offer: several trace types on one axis, a
secondary y-axis, or fine control over a single trace.

## 1.4 The three update methods

Almost all customisation goes through three methods, and knowing which is which saves a great
deal of searching.

| Method | Changes | Example |
|---|---|---|
| `update_layout` | The figure: title, size, legend, margins, hover mode | `fig.update_layout(title="Visits", height=400)` |
| `update_traces` | The data marks: marker size, line width, hover template | `fig.update_traces(marker_size=10)` |
| `update_xaxes` / `update_yaxes` | One axis: range, title, tick format, gridlines | `fig.update_yaxes(range=[0, 100])` |

## 1.5 Hover text

The default hover shows the plotted values. `hover_data` adds columns:

```python
px.scatter(df, x="attendance", y="gpa", hover_data=["student_id", "faculty"])
```

For full control, `hovertemplate` takes a format string:

```python
fig.update_traces(
    hovertemplate="<b>%{customdata[0]}</b><br>GPA %{y:.2f}<extra></extra>"
)
```

`<extra></extra>` suppresses the secondary box that otherwise shows the trace name.

Hover is the interaction most worth investing in, because it removes the need to read values
off an axis — which is the main thing a static chart is bad at.

## 1.6 Faceting and animation

`px` facets with one argument, as seaborn's figure-level functions do:

```python
px.scatter(df, x="attendance", y="gpa", facet_col="faculty", facet_col_wrap=2)
```

It also animates:

```python
px.scatter(df, x="gdp", y="life_expectancy", animation_frame="year")
```

Animation is the feature most often used badly. A reader cannot compare a frame with one they
saw four seconds ago, so an animated chart is weak for *comparison* and good only for showing
a *process* unfolding. Small multiples — a facet grid — beat animation whenever the question
is "how do these differ".

## 1.7 Saving and sharing

```python
fig.write_html("chart.html")                       # interactive, standalone
fig.write_html("chart.html", include_plotlyjs="cdn")   # smaller, needs internet
fig.write_image("chart.png", scale=2)              # static; needs the kaleido package
```

`include_plotlyjs=True` (the default) embeds the whole Plotly library, giving a file of about
3 MB that works offline. `"cdn"` loads it from the internet instead, giving a file of a few
kilobytes that needs a connection.

For a printed report you want `write_image`, which means installing `kaleido` — and at that
point you have a static chart, so matplotlib would have been simpler.

## 1.8 What a dashboard is

A dashboard is a set of linked views of the same data, usually with controls to filter it.
Building a real one needs a web framework — Dash, Streamlit or Panel — which is beyond this
course.

What is within reach is the idea: **several coordinated views answering one question**,
arranged so that each supports the others. The final section of Part 2 builds one as a single
HTML page. Your Week 14 project may include something similar.

A dashboard with twelve charts and no question is not a dashboard. It is a wall.

---

---

# Part 2 — Code example

The setup cell sets `pio.renderers.default = "notebook"`, which renders each figure inline with
its JavaScript embedded, so the charts are interactive and work without an internet
connection. Hover, zoom and legend-click all work as you read.

The cost is output size: the first figure embeds the Plotly library. If your saved notebook
becomes uncomfortably large, switch to `"notebook_connected"`, which loads the library from the
internet instead.

## 2.1 The data

In [ ]:
def generate_campus_data(seed: int = RANDOM_SEED) -> tuple:
    """Build a daily series and a student table for the charting examples."""
    generator = np.random.default_rng(seed)

    dates = pd.date_range("2026-01-01", "2026-12-31", freq="D")
    day_index = np.arange(len(dates))
    weekday = dates.dayofweek.to_numpy()

    exams = np.zeros(len(dates))
    for peak, height in [(150, 180), (330, 220)]:
        exams += height * np.exp(-((day_index - peak) ** 2) / 260)

    daily = pd.DataFrame({
        "date": dates,
        "visits": np.clip(
            260 + day_index * 0.42 + np.where(weekday >= 5, -130, 40)
            + exams + generator.normal(0, 22, len(dates)), 0, None
        ).round().astype(int),
        "is_weekend": weekday >= 5,
    })
    daily["month"] = daily["date"].dt.month_name()

    n = 300
    faculties = ["Engineering", "Business", "Science", "Arts"]
    faculty = generator.choice(faculties, size=n, p=[0.35, 0.30, 0.20, 0.15])
    ability = generator.normal(0, 1, size=n)

    students = pd.DataFrame({
        "student_id": [f"S{2026000 + i}" for i in range(n)],
        "faculty": faculty,
        "year_level": generator.choice([1, 2, 3, 4], size=n, p=[0.30, 0.30, 0.25, 0.15]),
        "gpa": np.clip(2.6 + ability * 0.5, 0, 4).round(2),
        "attendance_pct": np.clip(78 + ability * 9 + generator.normal(0, 5, n),
                                  0, 100).round(1),
        "study_hours": np.clip(generator.normal(9, 3.2, n) + ability * 1.1, 0, None).round(1),
    })

    return daily, students


daily, students = generate_campus_data()

print(f"daily    : {len(daily)} rows")
print(f"students : {len(students)} rows")
print(f"\nplotly {__import__('plotly').__version__}, "
      f"renderer = {pio.renderers.default}")
print(daily.head(3).to_string(index=False))

## 2.2 Plotly Express: one line

The same four chart types as Week 7, now interactive.

In [ ]:
fig = px.line(daily, x="date", y="visits",
              title="Library visits, 2026")
fig.update_layout(height=380, margin=dict(l=50, r=20, t=50, b=40))
fig.show()

print("One line of code. Hover, zoom and pan all work in a live notebook.")
print("\nThis is the Week 10 problem: 365 points at page width overlap into a")
print("band. In a static chart you would resample and lose the weekly cycle.")
print("Here the reader can zoom into a fortnight and see it.")

In [ ]:
# Scatter with a third variable as colour, and extra columns in the hover.
fig = px.scatter(
    students,
    x="attendance_pct", y="gpa",
    color="faculty",
    hover_data=["student_id", "study_hours"],
    title="Attendance against GPA, by faculty",
    labels={"attendance_pct": "attendance (%)", "gpa": "GPA"},
    color_discrete_sequence=PALETTE[:4],
)
fig.update_traces(marker=dict(size=7, opacity=0.7,
                              line=dict(width=0.5, color="white")))
fig.update_layout(height=430, legend_title_text="")
fig.show()

print("hover_data adds columns to the tooltip that are not plotted.")
print("\nThat is the thing a static chart genuinely cannot do: identify an")
print("individual point. In Week 7 an interesting outlier was anonymous —")
print("here the reader hovers and gets the student ID.")

In [ ]:
summary = students.groupby("faculty", as_index=False).agg(
    mean_gpa=("gpa", "mean"),
    n=("student_id", "count"),
).round(2)

fig = px.bar(
    summary.sort_values("mean_gpa"),
    x="mean_gpa", y="faculty", orientation="h",
    text="mean_gpa",
    title="Mean GPA by faculty",
    labels={"mean_gpa": "mean GPA", "faculty": ""},
    color_discrete_sequence=[PALETTE[0]],
    hover_data=["n"],
)
fig.update_traces(textposition="outside")
fig.update_xaxes(range=[0, 4])
fig.update_layout(height=330, showlegend=False)
fig.show()

print("Week 7's rules have not changed. The value axis still starts at zero,")
print("the bars are still sorted, the values are still labelled.")
print("\nInteractivity does not excuse you from any of the six decisions.")

## 2.3 Express and Graph Objects produce the same object

In [ ]:
express_figure = px.scatter(students, x="attendance_pct", y="gpa")

objects_figure = go.Figure()
objects_figure.add_trace(go.Scatter(
    x=students["attendance_pct"],
    y=students["gpa"],
    mode="markers",
))

print("px returns :", type(express_figure).__name__)
print("go returns :", type(objects_figure).__name__)
print("same class :", type(express_figure) is type(objects_figure))

print(f"\nA px figure is a normal Figure with {len(express_figure.data)} trace(s)")
print("in it. You can add traces to it, update it, or pull it apart.")
print("\nSo the practical approach is: build with px, adjust with the figure's")
print("own methods. Use go only for what px does not offer.")

In [ ]:
# Something px does not do: two chart types on one axis, with a second y-axis.
monthly = (daily.set_index("date")
           .resample("ME")
           .agg(total_visits=("visits", "sum"), mean_visits=("visits", "mean"))
           .reset_index())
monthly["label"] = monthly["date"].dt.strftime("%b")

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=monthly["label"], y=monthly["total_visits"], name="total visits",
           marker_color=PALETTE[0], opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=monthly["label"], y=monthly["mean_visits"], name="mean per day",
               mode="lines+markers", line=dict(color=PALETTE[3], width=3)),
    secondary_y=True,
)

fig.update_layout(title="Monthly totals and daily averages", height=400,
                  hovermode="x unified",
                  legend=dict(orientation="h", y=1.1))
fig.update_yaxes(title_text="total visits", secondary_y=False)
fig.update_yaxes(title_text="mean visits per day", secondary_y=True)
fig.show()

print("Bars and a line on one chart, with two different y-axes. px cannot")
print("build this; go can.")
print("\nA WARNING about the second axis. Two axes with different scales let")
print("you make any two series appear to move together or apart, just by")
print("choosing the ranges. Use it only when both quantities genuinely")
print("describe the same thing, and state both units clearly.")
print("\nhovermode='x unified' shows both values in one tooltip, which is what")
print("makes a two-axis chart readable at all.")

## 2.4 Faceting and animation

In [ ]:
fig = px.scatter(
    students, x="attendance_pct", y="gpa",
    facet_col="faculty", facet_col_wrap=2,
    color="faculty", color_discrete_sequence=PALETTE[:4],
    title="Attendance against GPA, one panel per faculty",
    labels={"attendance_pct": "attendance (%)", "gpa": "GPA"},
    height=520,
)
fig.update_traces(marker=dict(size=6, opacity=0.7))
fig.update_layout(showlegend=False)
# px writes "faculty=Engineering" in the facet titles; shorten them.
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

print("One argument, four panels, shared scales.")
print("\nShared scales are what makes the panels comparable — the same point")
print("as Week 7's sharex and sharey, and as Anscombe's quartet in Week 1.")

In [ ]:
# Animation: the feature most often used badly.
cumulative = []
for month in range(1, 13):
    upto = students.sample(frac=month / 12, random_state=month)
    snapshot = upto.groupby("faculty", as_index=False)["gpa"].mean()
    snapshot["month"] = month
    cumulative.append(snapshot)
frames = pd.concat(cumulative, ignore_index=True)

fig = px.bar(
    frames, x="faculty", y="gpa", animation_frame="month",
    range_y=[0, 4], color="faculty", color_discrete_sequence=PALETTE[:4],
    title="Mean GPA as more students are surveyed (animated)",
    height=400,
)
fig.update_layout(showlegend=False)
fig.show()

print("A play button appears in a live notebook.")
print("\nUSE THIS SPARINGLY. A reader cannot compare the current frame with one")
print("they saw four seconds ago, so animation is weak for COMPARISON.")
print("It is good for showing a PROCESS unfolding — an epidemic spreading, a")
print("city growing.")
print("\nWhenever the question is 'how do these differ', small multiples beat")
print("animation, because the reader can look back and forth.")

## 2.5 When a static chart is still better

The judgement this session is really about.

In [ ]:
comparison = pd.DataFrame([
    ("A printed report or PDF", "matplotlib",
     "interactive charts do not print"),
    ("A slide deck shown from a file", "matplotlib",
     "no browser, and nobody will interact mid-talk"),
    ("A figure for a journal article", "matplotlib",
     "publishers require static vector images"),
    ("Exploring data yourself", "either",
     "plotly's zoom is genuinely faster"),
    ("A shared HTML report", "plotly",
     "readers can drill in without asking you"),
    ("Dense time series a reader must inspect", "plotly",
     "zoom replaces resampling"),
    ("Many series a reader wants to compare selectively", "plotly",
     "legend click hides and shows"),
    ("One clear finding for an audience", "matplotlib",
     "interactivity adds nothing and may distract"),
    ("Millions of points", "matplotlib",
     "plotly embeds the data; the file becomes unusable"),
], columns=["situation", "use", "why"])

print(comparison.to_string(index=False))

print("\n" + "=" * 74)
print("THE RULE")
print("=" * 74)
print("  The chart must make sense BEFORE anyone touches it.")
print()
print("  Interactivity is for the reader who wants to go further. It is not a")
print("  substitute for a clear default view, and a finding that requires a")
print("  hover to be noticed has not been communicated.")

In [ ]:
# File size, measured rather than asserted.
from pathlib import Path

small = px.line(daily, x="date", y="visits", title="365 points")

full = Path("week11_full.html")
cdn = Path("week11_cdn.html")
small.write_html(full)                              # embeds all of plotly.js
small.write_html(cdn, include_plotlyjs="cdn")       # loads it from the internet

print(f"{'file':<36}{'size':>12}{'works offline?':>18}")
print("-" * 66)
print(f"{'write_html() default':<36}{full.stat().st_size / 1024:>10.0f} KB{'yes':>18}")
print(f"{'write_html(include_plotlyjs=cdn)':<36}{cdn.stat().st_size / 1024:>10.0f} KB{'no':>18}")

matplotlib_equivalent = 60
print(f"{'an equivalent matplotlib PNG':<36}{matplotlib_equivalent:>10} KB{'yes':>18}")

print("\nThe default embeds the whole Plotly library so the file works with no")
print("internet. 'cdn' makes it tiny and dependent on a connection.")
print("\nAnd the data itself is in the HTML. 365 points is nothing; a million")
print("rows produces a file no browser will open.")

for path in (full, cdn):
    path.unlink()
print("\n(temporary files removed)")

## 2.6 A small dashboard

Several coordinated views of one question, on a single page.

In [ ]:
def build_dashboard(daily_df: pd.DataFrame, student_df: pd.DataFrame) -> go.Figure:
    """Assemble four coordinated views into one figure.

    The question the dashboard answers: how busy is the library, and who
    is using it?

    Parameters
    ----------
    daily_df : pandas.DataFrame
        Daily visit counts.
    student_df : pandas.DataFrame
        One row per student.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    figure = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Daily visits, 7-day average",
            "Weekday against weekend",
            "GPA distribution",
            "Attendance against GPA",
        ),
        vertical_spacing=0.14,
        horizontal_spacing=0.10,
    )

    smoothed = daily_df.set_index("date")["visits"].rolling(7, center=True).mean()
    figure.add_trace(
        go.Scatter(x=smoothed.index, y=smoothed.to_numpy(), mode="lines",
                   line=dict(color=PALETTE[0], width=2), name="7-day mean",
                   showlegend=False),
        row=1, col=1,
    )

    by_type = daily_df.groupby("is_weekend")["visits"].mean()
    figure.add_trace(
        go.Bar(x=["weekday", "weekend"],
               y=[by_type[False], by_type[True]],
               marker_color=[PALETTE[0], PALETTE[3]], showlegend=False,
               text=[f"{by_type[False]:.0f}", f"{by_type[True]:.0f}"],
               textposition="outside"),
        row=1, col=2,
    )

    figure.add_trace(
        go.Histogram(x=student_df["gpa"], nbinsx=22,
                     marker_color=PALETTE[2], showlegend=False),
        row=2, col=1,
    )

    figure.add_trace(
        go.Scatter(x=student_df["attendance_pct"], y=student_df["gpa"],
                   mode="markers",
                   marker=dict(size=5, color=PALETTE[4], opacity=0.55),
                   text=student_df["student_id"],
                   hovertemplate="%{text}<br>attendance %{x:.1f}%"
                                 "<br>GPA %{y:.2f}<extra></extra>",
                   showlegend=False),
        row=2, col=2,
    )

    figure.update_yaxes(title_text="visits", row=1, col=1)
    figure.update_yaxes(title_text="mean visits", row=1, col=2)
    figure.update_yaxes(title_text="students", row=2, col=1)
    figure.update_yaxes(title_text="GPA", row=2, col=2)
    figure.update_xaxes(title_text="GPA", row=2, col=1)
    figure.update_xaxes(title_text="attendance (%)", row=2, col=2)

    figure.update_layout(
        height=700,
        title_text="Library use, 2026",
        hovermode="closest",
        margin=dict(l=60, r=30, t=80, b=50),
    )
    return figure


dashboard = build_dashboard(daily, students)
dashboard.show()

print("Four views, one question: how busy is the library, and who uses it?")
print("\nEach panel earns its place by answering part of that question. A")
print("dashboard with twelve charts and no question is not a dashboard.")
print("It is a wall.")

---

# Part 3 — Guided workshop (in class)

Four tasks. Each has a verification cell that inspects the figure object you built.

### Task 1 — A first interactive chart

**Problem.** Build `fig1`: a Plotly Express line chart of `visits` against `date` from
`daily`, with a title and axis labels.

**Hints**

- `px.line(daily, x="date", y="visits", title="...")`.
- `labels={"visits": "visits per day"}` renames an axis.

**Expected output.** A line chart, followed by `Task 1 complete`.

In [ ]:
# TODO 1: a px.line of visits against date, with a title.
fig1 = None

if fig1 is not None:
    fig1.show()

In [ ]:
# Verification for Task 1.
problems = []
if fig1 is None:
    problems.append("fig1 is still None")
elif not isinstance(fig1, go.Figure):
    problems.append(f"expected a plotly Figure, got {type(fig1).__name__}")
else:
    if len(fig1.data) == 0:
        problems.append("the figure has no traces")
    elif fig1.data[0].type != "scatter":
        problems.append(f"expected a line chart (scatter trace), "
                        f"got {fig1.data[0].type}")
    elif len(fig1.data[0].x) != len(daily):
        problems.append(f"expected {len(daily)} points, got {len(fig1.data[0].x)}")
    if not (fig1.layout.title.text or ""):
        problems.append("no title — pass title= to px.line")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 1 complete.")

### Task 2 — Colour, hover and styling

**Problem.** Build `fig2`: a scatter of `study_hours` (x) against `gpa` (y) from `students`,
with:

- points coloured by `faculty`,
- `student_id` and `attendance_pct` available on hover,
- a title,
- a figure height of 450.

**Hints**

- `color="faculty"` and `hover_data=["student_id", "attendance_pct"]`.
- `fig2.update_layout(height=450)`.
- Four faculties means four traces, one per colour.

**Expected output.** A coloured scatter, followed by `Task 2 complete`.

In [ ]:
# TODO 2: scatter coloured by faculty, with extra hover columns and height 450.
fig2 = None

if fig2 is not None:
    fig2.show()

In [ ]:
# Verification for Task 2.
problems = []
if fig2 is None:
    problems.append("fig2 is still None")
elif not isinstance(fig2, go.Figure):
    problems.append(f"expected a plotly Figure, got {type(fig2).__name__}")
else:
    if len(fig2.data) != students["faculty"].nunique():
        problems.append(f"expected {students['faculty'].nunique()} traces "
                        f"(one per faculty), got {len(fig2.data)} — pass color='faculty'")
    if not (fig2.layout.title.text or ""):
        problems.append("no title")
    if fig2.layout.height != 450:
        problems.append(f"height should be 450, got {fig2.layout.height}")
    has_custom = any(trace.customdata is not None for trace in fig2.data)
    if not has_custom:
        problems.append("no extra hover columns — pass hover_data=[...]")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print(f"Task 2 complete. {len(fig2.data)} traces, one per faculty.")

### Task 3 — Graph Objects, for something `px` will not do

**Problem.** Build `fig3` with Graph Objects: a single chart showing **both**

- a bar trace of mean `gpa` by `faculty`, and
- a scatter trace of the **count** of students per faculty, as markers.

Both on the same x-axis. Give the figure a title.

**Hints**

- `go.Figure()`, then `add_trace(go.Bar(...))` and `add_trace(go.Scatter(..., mode="markers"))`.
- Build the summary with `groupby` first.
- Two traces of different types is exactly what `px` cannot do in one call.

**Expected output.** A chart with two traces, followed by `Task 3 complete`.

In [ ]:
faculty_summary = students.groupby("faculty", as_index=False).agg(
    mean_gpa=("gpa", "mean"),
    n=("student_id", "count"),
)
print(faculty_summary.round(2).to_string(index=False))

# TODO 3: a go.Figure with a Bar trace and a Scatter trace, plus a title.
fig3 = None

if fig3 is not None:
    fig3.show()

In [ ]:
# Verification for Task 3.
problems = []
if fig3 is None:
    problems.append("fig3 is still None")
elif not isinstance(fig3, go.Figure):
    problems.append(f"expected a plotly Figure, got {type(fig3).__name__}")
else:
    types = sorted(trace.type for trace in fig3.data)
    if len(fig3.data) != 2:
        problems.append(f"expected exactly 2 traces, got {len(fig3.data)}")
    elif types != ["bar", "scatter"]:
        problems.append(f"expected one bar and one scatter trace, got {types}")
    if not (fig3.layout.title.text or ""):
        problems.append("no title")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 3 complete. Two trace types on one figure — px cannot do this.")

### Task 4 — Decide when not to use Plotly

**Problem.** For each situation, set the value to `"plotly"` or `"matplotlib"`.

| Key | Situation |
|---|---|
| `"printed_report"` | A figure for a printed annual report |
| `"shared_html"` | A weekly HTML summary emailed to colleagues |
| `"conference_slide"` | A slide for a conference talk, shown from a PDF |
| `"dense_timeseries"` | Five years of hourly readings a colleague needs to inspect |

**Hints**

- Ask one question of each: will anyone be able to interact with this?
- If the answer is no, interactivity is only a cost.

**Expected output.** Four answers, followed by `Task 4 complete`.

In [ ]:
# TODO 4: "plotly" or "matplotlib" for each.
choices = {
    "printed_report": None,
    "shared_html": None,
    "conference_slide": None,
    "dense_timeseries": None,
}

for key, value in choices.items():
    print(f"  {key:<20} -> {value}")

In [ ]:
# Verification for Task 4.
correct = {
    "printed_report": "matplotlib",
    "shared_html": "plotly",
    "conference_slide": "matplotlib",
    "dense_timeseries": "plotly",
}
reasons = {
    "printed_report": "paper cannot be hovered over or zoomed",
    "shared_html": "readers can drill in without asking you",
    "conference_slide": "a PDF has no browser, and nobody interacts mid-talk",
    "dense_timeseries": "zoom replaces resampling, so nothing is discarded",
}

problems = []
for key, want in correct.items():
    given = choices.get(key)
    if given is None:
        problems.append(f"{key!r} is still None")
    elif str(given).lower() != want:
        problems.append(f"{key!r}: {given!r} — {want} is better, because "
                        f"{reasons[key]}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 4 complete.")
    print("\nThe question to ask every time: will anyone actually interact with")
    print("this? If not, interactivity is pure cost.")

---

# Part 4 — Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 — A reusable interactive time-series chart

**Problem.** Write `plot_timeseries(df, date_col, value_col, smooth=None, title=None)`.

**Requirements.** Return a Plotly `Figure` with:

| Requirement | Detail |
|---|---|
| Raw series | One `scatter` trace in `lines` mode, named `"daily"` |
| `smooth=None` | Only the raw trace |
| `smooth=n` | A **second** trace of the centred rolling mean over `n` periods, named `"n-period mean"` |
| Title | `title` if given, otherwise `"<value_col> over time"` |
| Axis titles | x-axis titled with `date_col`, y-axis with `value_col` |
| Hover | `hovermode="x unified"` |
| Height | 420 |

**Requirements on errors**

- Raise `KeyError` if either column is absent.
- Raise `ValueError` if `smooth` is given and is less than 2.
- Raise `TypeError` if `date_col` is not a datetime column.

Do not modify the input, and do not call `.show()`.

**Hints**

- `go.Figure()` then `add_trace(go.Scatter(x=..., y=..., mode="lines", name=...))`.
- `series.rolling(n, center=True).mean()` — Week 10.
- `fig.update_layout(title=..., hovermode="x unified", height=420)`.
- `fig.update_xaxes(title_text=...)` and `update_yaxes`.
- `pd.api.types.is_datetime64_any_dtype` for the type check.

In [ ]:
# Problem 1 — starter code.

def plot_timeseries(df, date_col: str, value_col: str,
                    smooth=None, title=None) -> go.Figure:
    """Build an interactive time-series chart, optionally with a smoothed overlay.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    date_col, value_col : str
        Column names.
    smooth : int or None
        Window for a centred rolling mean, drawn as a second trace.
    title : str or None
        Figure title. Defaults to "<value_col> over time".

    Returns
    -------
    plotly.graph_objects.Figure

    Raises
    ------
    KeyError
        If either column is absent.
    ValueError
        If smooth is given and is less than 2.
    TypeError
        If date_col is not a datetime column.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete plot_timeseries()")


demo = plot_timeseries(daily, "date", "visits", smooth=7,
                       title="Library visits with a 7-day mean")
demo.show()

In [ ]:
# Problem 1 — validation. All assertions must pass.

sample = pd.DataFrame({
    "when": pd.date_range("2026-01-01", periods=60, freq="D"),
    "amount": np.linspace(10, 70, 60),
    "label": list("abcde") * 12,
})

# --- without smoothing ---
plain = plot_timeseries(sample, "when", "amount")

assert isinstance(plain, go.Figure), "Must return a plotly Figure"
assert len(plain.data) == 1, (
    f"With smooth=None there should be exactly one trace; got {len(plain.data)}"
)
assert plain.data[0].mode == "lines", "The raw series should be drawn in lines mode"
assert plain.data[0].name == "daily", (
    f"The raw trace should be named 'daily'; got {plain.data[0].name!r}"
)
assert len(plain.data[0].x) == 60

# --- default title and axis titles ---
assert plain.layout.title.text == "amount over time", (
    f"Default title should be '<value_col> over time'; got {plain.layout.title.text!r}"
)
assert plain.layout.xaxis.title.text == "when"
assert plain.layout.yaxis.title.text == "amount"
assert plain.layout.hovermode == "x unified"
assert plain.layout.height == 420

# --- an explicit title wins ---
titled = plot_timeseries(sample, "when", "amount", title="Custom")
assert titled.layout.title.text == "Custom"

# --- with smoothing ---
smoothed = plot_timeseries(sample, "when", "amount", smooth=7)
assert len(smoothed.data) == 2, (
    f"With smooth=7 there should be two traces; got {len(smoothed.data)}"
)
names = [trace.name for trace in smoothed.data]
assert "daily" in names, "The raw trace must still be present"
assert "7-period mean" in names, (
    f"The smoothed trace should be named '7-period mean'; got {names}"
)

smooth_trace = [t for t in smoothed.data if t.name == "7-period mean"][0]
reference = sample["amount"].rolling(7, center=True).mean()
assert np.allclose(np.asarray(smooth_trace.y, dtype=float),
                   reference.to_numpy(), equal_nan=True), (
    "The smoothed trace must be a CENTRED rolling mean"
)

# --- errors ---
for bad_date, bad_value in [("nope", "amount"), ("when", "nope")]:
    try:
        plot_timeseries(sample, bad_date, bad_value)
    except KeyError:
        pass
    else:
        raise AssertionError(f"Missing column ({bad_date}, {bad_value}) must raise KeyError")

for bad_window in [1, 0, -3]:
    try:
        plot_timeseries(sample, "when", "amount", smooth=bad_window)
    except ValueError:
        pass
    else:
        raise AssertionError(f"smooth={bad_window} must raise ValueError")

try:
    plot_timeseries(sample, "label", "amount")
except TypeError:
    pass
else:
    raise AssertionError("A non-datetime date_col must raise TypeError")

# --- the input is untouched ---
before = sample.copy()
_ = plot_timeseries(sample, "when", "amount", smooth=5)
assert sample.equals(before), "plot_timeseries must not modify its input"

print("Problem 1: all assertions passed.")

---

## Problem 2 — A two-panel summary figure

**Problem.** Write `build_faculty_panel(df, group_col, value_col)`.

It returns a `Figure` with **two panels side by side** summarising one numeric column by one
grouping column.

**The two panels**

| Position | Content |
|---|---|
| Left (row 1, col 1) | A **bar** trace of the mean of `value_col` per group, sorted **descending** |
| Right (row 1, col 2) | A **box** trace of `value_col` per group, showing the full distribution |

**Requirements**

- Use `make_subplots(rows=1, cols=2, subplot_titles=(...))` with a title for each panel.
- The figure has an overall `title`.
- Height 420.
- The left panel's y-axis starts at 0 — the Week 7 rule still applies.
- Both panels' y-axes are titled with `value_col`.
- Return the `Figure`; do not call `.show()`.

**Requirements on errors**

- Raise `KeyError` if either column is absent.
- Raise `TypeError` if `value_col` is not numeric.

**Hints**

- `figure.add_trace(go.Bar(...), row=1, col=1)`.
- For the box panel, add **one `go.Box` trace per group**, each with `y=` the group's values
  and `name=` the group label. That is how Plotly builds a grouped box plot from Graph
  Objects.
- `figure.update_yaxes(range=[0, ...], row=1, col=1)` for the zero baseline.
- The two panels say different things: the bar gives the summary, the box gives the shape.
  That pairing is the Week 8 lesson.

In [ ]:
# Problem 2 — starter code.

def build_faculty_panel(df, group_col: str, value_col: str) -> go.Figure:
    """Build a two-panel figure: mean by group, and distribution by group.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    group_col : str
        Column holding the groups.
    value_col : str
        Numeric column to summarise.

    Returns
    -------
    plotly.graph_objects.Figure

    Raises
    ------
    KeyError
        If either column is absent.
    TypeError
        If value_col is not numeric.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete build_faculty_panel()")


panel = build_faculty_panel(students, "faculty", "gpa")
panel.show()

In [ ]:
# Problem 2 — validation. All assertions must pass.

sample = pd.DataFrame({
    "team": ["A"] * 20 + ["B"] * 20 + ["C"] * 20,
    "score": np.concatenate([
        np.linspace(10, 30, 20),
        np.linspace(50, 70, 20),
        np.linspace(20, 40, 20),
    ]),
    "note": ["x"] * 60,
})
# means: A = 20, B = 60, C = 30  ->  descending order is B, C, A

fig = build_faculty_panel(sample, "team", "score")

assert isinstance(fig, go.Figure), "Must return a plotly Figure"

bars = [t for t in fig.data if t.type == "bar"]
boxes = [t for t in fig.data if t.type == "box"]

assert len(bars) == 1, f"Expected exactly one bar trace; got {len(bars)}"
assert len(boxes) == 3, (
    f"Expected one box trace per group (3); got {len(boxes)}. Add a go.Box per "
    "group, each with its own y= and name=."
)

# --- the bar panel ---
bar = bars[0]
assert len(bar.y) == 3, f"One bar per group; got {len(bar.y)}"
assert np.allclose(np.asarray(bar.y, dtype=float), [60.0, 30.0, 20.0]), (
    f"Bars must be sorted descending by mean: B=60, C=30, A=20. Got {list(bar.y)}"
)
assert list(bar.x) == ["B", "C", "A"], (
    f"Bar categories must follow the sorted order; got {list(bar.x)}"
)

# --- the zero baseline, from Week 7 ---
left_range = fig.layout.yaxis.range
assert left_range is not None, (
    "The left panel's y-axis range must be set explicitly so it starts at 0"
)
assert np.isclose(left_range[0], 0), (
    f"The bar panel's y-axis must start at 0; it starts at {left_range[0]}"
)

# --- the box panel covers every group ---
box_names = sorted(trace.name for trace in boxes)
assert box_names == ["A", "B", "C"], f"Box traces should be named per group; got {box_names}"

# --- titles ---
assert (fig.layout.title.text or ""), "The figure needs an overall title"
assert fig.layout.annotations, "Use subplot_titles to label the two panels"
assert len(fig.layout.annotations) >= 2, (
    f"Expected two subplot titles; found {len(fig.layout.annotations)}"
)

assert fig.layout.height == 420, f"Height should be 420; got {fig.layout.height}"

# --- axis titles ---
assert fig.layout.yaxis.title.text == "score", (
    f"Left y-axis should be titled with value_col; got {fig.layout.yaxis.title.text!r}"
)
assert fig.layout.yaxis2.title.text == "score", (
    f"Right y-axis should be titled with value_col; got {fig.layout.yaxis2.title.text!r}"
)

# --- errors ---
for bad_group, bad_value in [("nope", "score"), ("team", "nope")]:
    try:
        build_faculty_panel(sample, bad_group, bad_value)
    except KeyError:
        pass
    else:
        raise AssertionError(f"Missing column ({bad_group}, {bad_value}) must raise KeyError")

try:
    build_faculty_panel(sample, "team", "note")
except TypeError:
    pass
else:
    raise AssertionError("A non-numeric value_col must raise TypeError")

# --- the input is untouched ---
before = sample.copy()
_ = build_faculty_panel(sample, "team", "score")
assert sample.equals(before), "The function must not modify its input"

# --- it works on the notebook's own data ---
real = build_faculty_panel(students, "faculty", "attendance_pct")
assert len([t for t in real.data if t.type == "box"]) == students["faculty"].nunique()

print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 120–200 words.

1. Section 2.5 argued that a chart must make sense before anyone touches it. Explain why, in
   terms of what actually happens when a reader opens a report.
2. Give one situation from your own experience — a report, a presentation, a website — where
   an interactive chart would have been better than a static one, and one where it would have
   been worse. Say why in each case.
3. Week 10 showed that resampling a year of daily data to monthly destroys the weekly cycle.
   Explain how interactivity offers a different answer to that problem, and name one thing
   that answer still does not solve.

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 120–200 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 12

Week 12 is statistical analysis: distributions, sampling, and hypothesis testing. It gives you
the vocabulary to say whether a difference you have plotted is larger than you would expect
from chance alone.

Before the session, read Pfaffelhuber §9.2–9.3 and McKinney §12.3.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §9.3.

Plotly Technologies Inc. *Plotly Express documentation*.
Available at `https://plotly.com/python/plotly-express/`.